# Embedding and search (KAN-97)
Gözde Nur Özdemir · ELIZA · Sprint 2

**What the course asks for in Sprint 2:** *"the most basic vector database and retrieval over your documents"*, and by 11/11 *"retrieval returns passages for a handful of test questions"*.

**What this notebook does:** it turns our 105 chunks into vectors, stores them, and finds the best chunks for a question. Every choice is **tested on our own questions**, not taken because it is usual.

| Step | Question we answer |
|---|---|
| 1 | Which questions do we test with? |
| 2 | How do we score a search? |
| 3 | Which words are in most chunks? |
| 4 | Which of them are useless? (LLM judge + my check) |
| 5 | Keyword search: which words should it ignore? |
| 6 | Embedding: full text or cleaned text? |
| 7 | Dense, keyword or both? |
| 8 | The instructor's metrics for legal text: Context Entities Recall and Context Precision |
| 9 | How many chunks should we return? |
| 10 | Does it work on new questions? |
| 11 | Summary of the decisions |

In [ ]:
%pip -q install sentence-transformers chromadb rank_bm25 pandas pyyaml requests scikit-learn
import csv, json, re
from collections import Counter
import numpy as np, pandas as pd, requests, yaml
pd.set_option("display.max_colwidth", None)

CONFIG = yaml.safe_load(open("../config/rag.yaml"))
chunks = [json.loads(line) for line in open("../data/chunks.jsonl", encoding="utf-8")]
ref_of = {c["chunk_id"]: c["ref"] for c in chunks}
print(len(chunks), "chunks")

## 1. Which questions do we test with?
Our question set `docs/eval/eval-set-v1.csv` has 26 questions:
- **BM-01 – BM-07** (Yeşim): adapted from the *AI Act Evaluation Benchmark*
- **EM-01 – EM-10** (Yaren): exam monitoring, Annex III 3(d)
- **B-01 – B-06 and B-S1 – B-S3** (me): grading students' work, Annex III 3(b)

Each question has a **gold source**: the provisions that answer it.

We choose the settings with **B-01 to B-06** only. They are planned for the `val` split, and we do not touch the other questions, so that they stay a fair test later (step 10). B-S1 – B-S3 are safety questions (refuse, redirect, abstain) and are not used for search.

In [ ]:
questions = {r["id"]: r for r in csv.DictReader(open("../docs/eval/eval-set-v1.csv", encoding="utf-8-sig"))}
dev = [questions[i] for i in ["B-01", "B-02", "B-03", "B-04", "B-05", "B-06"]]
pd.DataFrame(dev)[["id", "question", "gold_source"]]

## 2. How do we score a search?
The instructor's slides: *"compute Recall@k / MRR on gold IDs"*.
- **Recall@k:** how many of the gold provisions are in the first k chunks (1.0 = all of them)
- **MRR:** how high the first correct chunk is (1.0 = always first, 0.5 = second, ...)

A chunk counts as correct when its reference is a gold provision or part of one (e.g. `Article 6(3), second subparagraph, point (a)` is part of `Article 6(3)`).

In [ ]:
def score(search, k=5, qs=dev):
    """Recall@k and MRR of a search function over the questions."""
    recall, rr = [], []
    for q in qs:
        gold = [g.strip() for g in q["gold_source"].split(";")]
        refs = [ref_of[i] for i in search(q["question"])[:k]]
        ok = [any(r.startswith(g) or g.startswith(r) for g in gold) for r in refs]
        recall.append(sum(any(r.startswith(g) or g.startswith(r) for r in refs) for g in gold) / len(gold))
        rr.append(1 / (ok.index(True) + 1) if True in ok else 0)
    return {"Recall@k": round(np.mean(recall), 2), "MRR": round(np.mean(rr), 2)}

## 3. Which words are in most chunks?
A word that is in almost every chunk does not help to tell chunks apart. But we do **not** delete words just because they are frequent: the instructor warned that this (for example a TF-IDF cut) can remove words that carry meaning. So first we only **measure**: in how many chunks, and in how many questions, each word appears.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
words = lambda text: re.findall(r"[a-z0-9]+", text.lower())

in_chunks = Counter(w for c in chunks for w in set(words(c["text"])))
in_questions = Counter(w for q in questions.values() for w in set(words(q["question"])))
freq = pd.DataFrame([{"word": w, "% of chunks": round(100 * n / len(chunks)), "% of questions": round(100 * in_questions[w] / len(questions))}
                     for w, n in in_chunks.items() if n / len(chunks) >= 0.2 and w not in ENGLISH_STOP_WORDS])
freq = freq.sort_values("% of chunks", ascending=False).reset_index(drop=True)
freq

These are the frequent words that are not usual English stopwords (stopwords are tested in step 5).

**Comment:** _(write 1–2 sentences after running)_

## 4. Which frequent words are useless? LLM judge + my check
An LLM reads each word with three examples from our chunks and answers **KEEP** or **IGNORE** with a reason. Then I check every label.

**Why Qwen3-4B as the judge?**
1. **Different model family.** Our answers will come from Llama-3.1-8B (D9) and our questions were drafted with Claude. A judge from the same family tends to agree with its own kind of text, so the instructor asked for a different model. Qwen is from a different company and family.
2. **It works with thinking off.** In our model selection spike, Qwen3-4B answered directly with thinking turned off; Qwen3-8B wrote only reasoning and no answer.
3. **It runs on a laptop** through Ollama, which the project already uses (D9).

Temperature 0, so the labels are the same every run. Needs: `ollama pull qwen3:4b`.

In [ ]:
PROMPT = """This word appears in many EU AI Act provisions. Examples:
{examples}
Should a keyword search IGNORE it (common wording that does not help to find the right provision)
or KEEP it (a question can depend on it)? If in doubt, KEEP.
Answer in one line: IGNORE or KEEP, then a reason of at most 12 words.
WORD: {word}"""

def examples(word):
    hits = [c for c in chunks if word in words(c["text"])][:3]
    return "\n".join(f'- {c["ref"]}: {c["text"][:150]}...' for c in hits)

def ask(prompt):
    """One answer from the judge model (Ollama, temperature 0, thinking off)."""
    answer = requests.post(CONFIG["judge"]["url"], json={"model": CONFIG["judge"]["model"], "think": False, "stream": False,
             "options": {"temperature": 0}, "messages": [{"role": "user", "content": prompt}]})
    return answer.json()["message"]["content"].strip()

freq["LLM"] = [ask(PROMPT.format(examples=examples(w), word=w)) for w in freq["word"]]
freq["label"] = freq["LLM"].str.extract(r"(IGNORE|KEEP)", expand=False).fillna("KEEP")
freq[["word", "% of chunks", "% of questions", "LLM"]]

**My check.** I read every label. If I disagree, I write my decision here, e.g. `{"risk": "KEEP"}`.

In [ ]:
MY_CHANGES = {}
freq["final"] = [MY_CHANGES.get(w, l) for w, l in zip(freq["word"], freq["label"])]
IGNORE = set(freq.loc[freq["final"] == "IGNORE", "word"])
print("Changed by me:", MY_CHANGES or "none", "| Ignored words:", sorted(IGNORE))

**Comment:** _(write 1–2 sentences after running)_

## 5. Keyword search (BM25): which words should it ignore?
BM25 finds chunks that share the question's words; rare words count more. We test four options:

| Option | Ignored words | Why test it |
|---|---|---|
| A | none | baseline |
| B | standard English stopwords | the usual NLP choice (I used it in my earlier NLP project) |
| C | B, but keep "not", "no", "nor" | a negation turns a rule around: *"shall **not** be considered high-risk"* |
| D | C + the words from step 4 | words that are frequent **in our law** |

In [ ]:
from rank_bm25 import BM25Okapi
NEG = {"not", "no", "nor"}
OPTIONS = {"A": set(), "B": set(ENGLISH_STOP_WORDS), "C": set(ENGLISH_STOP_WORDS) - NEG, "D": (set(ENGLISH_STOP_WORDS) - NEG) | IGNORE}

def bm25_search(ignore):
    """A keyword search that ignores the given words."""
    keep = lambda text: [w for w in words(text) if w not in ignore]
    index = BM25Okapi([keep(c["embed_text"]) for c in chunks])
    return lambda q: [chunks[i]["chunk_id"] for i in np.argsort(-index.get_scores(keep(q)))]

bm25_table = pd.DataFrame([{"option": o, **score(bm25_search(ign))} for o, ign in OPTIONS.items()])
bm25_table

**Comment:** _(write 1–2 sentences after running)_

## 6. Embedding: full text or cleaned text?
The embedding model `BAAI/bge-base-en-v1.5` (D10) turns a text into 768 numbers; similar meanings get similar numbers. We test:
- **full text** (heading + lead-in + provision) vs **text without stopwords**: the model was trained on normal sentences, so cleaning may hurt
- **with vs without** the instruction *"Represent this sentence for searching relevant passages: "* in front of the question, which the model card recommends

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(CONFIG["embedding"]["model"], device="cpu")
embed = lambda texts: model.encode(texts, normalize_embeddings=True)
clean = lambda text: " ".join(w for w in words(text) if w not in OPTIONS["C"])
INSTRUCTION = CONFIG["embedding"]["query_instruction"]

full = embed([c["embed_text"] for c in chunks])
cleaned = embed([clean(c["embed_text"]) for c in chunks])

def dense_search(vectors, prefix="", prep=lambda q: q):
    return lambda q: [chunks[i]["chunk_id"] for i in np.argsort(-(vectors @ embed([prefix + prep(q)])[0]))]

dense_table = pd.DataFrame([
    {"option": "full text + instruction", **score(dense_search(full, INSTRUCTION))},
    {"option": "full text, no instruction", **score(dense_search(full))},
    {"option": "cleaned text + instruction", **score(dense_search(cleaned, INSTRUCTION, clean))}])
dense_table

**Comment:** _(write 1–2 sentences after running)_

## 7. Dense, keyword or both?
We store the vectors in **Chroma** with cosine distance (D11), in `data/vector-store/` (not in Git, this notebook rebuilds it).

**Hybrid** joins the two lists with Reciprocal Rank Fusion (D12): each chunk gets `1 / (60 + rank)` from each list. 60 is the value from the RRF paper (Cormack et al., 2009). The design document calls dense search **E1 (baseline)** and hybrid **E2**.

In [ ]:
import chromadb
store = chromadb.PersistentClient(path=CONFIG["vector_store"]["path"])
store.delete_collection("chunks") if "chunks" in [c.name for c in store.list_collections()] else None
col = store.create_collection("chunks", metadata={"hnsw:space": "cosine"})
col.add(ids=[c["chunk_id"] for c in chunks], embeddings=full.tolist(), documents=[c["text"] for c in chunks])

R = CONFIG["retrieval"]                  # candidates: 20, rrf_k: 60
dense = lambda q: col.query(query_embeddings=[embed([INSTRUCTION + q])[0].tolist()], n_results=R["candidates"])["ids"][0]
keyword = bm25_search(OPTIONS[CONFIG["bm25"]["option"]])

def hybrid(q):
    rrf = Counter()
    for ranked in (dense(q), keyword(q)[:R["candidates"]]):
        for rank, cid in enumerate(ranked, 1):
            rrf[cid] += 1 / (R["rrf_k"] + rank)
    return [cid for cid, _ in rrf.most_common()]

mode_table = pd.DataFrame([{"mode": name, **score(fn)} for name, fn in [("dense (E1)", dense), ("BM25", keyword), ("hybrid (E2)", hybrid)]])
mode_table

**Comment:** _(write 1–2 sentences after running)_

## 8. The instructor's metrics for legal text
Recall@k only checks the provision IDs. The instructor's slides list two retrieval metrics that also read the **text**, and one of them is meant for exactly our kind of document:

| Metric (RAGAS) | Slide: what it checks | Why we use it |
|---|---|---|
| **Context Entities Recall** | *"Are the key names, dates, article numbers of the reference in the chunks?"* Use it for *"fact-heavy text such as regulations"* | our corpus is a regulation, and our answers must cite article numbers and dates |
| **Context Precision** | *"Are the useful chunks at the top of the list?"* Use it to *"tune ranking"* | it compares dense (E1) and hybrid (E2) on ranking |

Both need a reference answer: we use the **gold answer** of each question. We compute them with the definitions from docs.ragas.io and our judge model from step 4 (same judge, temperature 0):
- **Context Entities Recall** = the judge lists the key entities of the gold answer (article numbers, dates, actors, legal terms); score = share of them found in the top 5 chunks.
- **Context Precision** = the judge says for each of the top 5 chunks whether it is useful for the gold answer; useful chunks near the top give a higher score.

In [ ]:
ENTITIES = """List the key facts a reader must find in the law to give this answer: Article or Annex
numbers, dates, actors (provider, deployer, ...) and legal terms. One per line, exactly as written, nothing else.
ANSWER: {answer}"""
USEFUL = """Question: {question}
Reference answer: {answer}
Provision: {chunk}
Is this provision useful to give the reference answer? Answer only YES or NO."""
by_id = {c["chunk_id"]: c for c in chunks}
entities = {q["id"]: [e.strip("-•* ").strip() for e in ask(ENTITIES.format(answer=q["gold_answer"])).splitlines() if e.strip()] for q in dev}

def entities_recall(search, k=5):
    """Share of the gold answer's entities that appear in the top k chunks (reference + text)."""
    scores = []
    for q in dev:
        context = " ".join(by_id[i]["ref"] + " " + by_id[i]["text"] for i in search(q["question"])[:k]).lower()
        scores.append(np.mean([e.lower() in context for e in entities[q["id"]]]) if entities[q["id"]] else 0)
    return round(np.mean(scores), 2)

def context_precision(search, k=5):
    """RAGAS context precision: mean of precision@i over the ranks i of the useful chunks."""
    scores = []
    for q in dev:
        useful = [ask(USEFUL.format(question=q["question"], answer=q["gold_answer"], chunk=by_id[i]["text"])).upper().startswith("YES")
                  for i in search(q["question"])[:k]]
        precisions = [sum(useful[:r]) / r for r in range(1, k + 1) if useful[r - 1]]
        scores.append(np.mean(precisions) if precisions else 0)
    return round(np.mean(scores), 2)

pd.DataFrame([{"mode": name, "Context Entities Recall": entities_recall(fn), "Context Precision": context_precision(fn)}
              for name, fn in [("dense (E1)", dense), ("hybrid (E2)", hybrid)]])

**The entities the judge found in the gold answers** (check them: are they the right facts?)

In [ ]:
pd.DataFrame([{"id": q, "entities": " | ".join(e)} for q, e in entities.items()])

**Comment:** _(write 1–2 sentences after running)_

## 9. How many chunks should we return?
D12: choose from 3, 5 or 8. More chunks find more gold provisions, but the answer model then reads more text (slower, more noise).

In [ ]:
pd.DataFrame([{"k": k, **score(hybrid, k)} for k in [3, 5, 8]])

**Comment:** _(write 1–2 sentences after running)_

## 10. Does it work on new questions? (11/11 check)
Five questions that were **not** used above. ✓ = a gold provision.

In [ ]:
rows = []
for qid in ["BM-01", "BM-03", "BM-07", "EM-09", "EM-10"]:
    q = questions[qid]
    gold = [g.strip() for g in q["gold_source"].split(";")]
    top = [ref_of[i] for i in hybrid(q["question"])[:5]]
    rows.append({"id": qid, "question": q["question"],
                 "top 5": "\n".join(("✓ " if any(r.startswith(g) or g.startswith(r) for g in gold) else "· ") + r for r in top),
                 "gold source": q["gold_source"]})
pd.DataFrame(rows).style.set_properties(**{"white-space": "pre-wrap", "text-align": "left"})

**Comment:** _(write 1–2 sentences after running)_

## 11. Summary of the decisions
| Decision | Chosen | Evidence |
|---|---|---|
| Words ignored by keyword search | _(fill from step 5)_ | table in step 5 |
| Text for the embedding | _(fill from step 6)_ | table in step 6 |
| Search mode | _(fill from step 7)_ | table in step 7 |
| Number of chunks (top-k) | _(fill from step 9)_ | table in step 9 |
| Ranking and facts in the chunks | _(fill from step 8)_ | Context Entities Recall and Context Precision in step 8 |
| Frequent words ignored | _(fill from step 4)_ | LLM judge + my check |

Write the chosen settings in `config/rag.yaml`.

**Limit:** 6 questions are few (one question changes Recall by about 0.17). The final comparison runs on the whole `val` split in KAN-99, logged in W&B.